# Data Preparation Pipelines for Knee Replacement Outcomes

## Key Workflow:
1. **Impute Q-question items** (pre-op and post-op questions 1-12)
2. **Calculate Q-Scores** (sum of items)
   - Pre-Op Q Score = Sum of Pre-Op Q questions 1-12
   - Post-Op Q Score = Sum of Post-Op Q questions 1-12
3. **Create target variable** = Post-Op Q Score - Pre-Op Q Score (improvement)
4. **Apply three imputation pipelines** to the prepared data

Each pipeline produces train/test datasets ready for modeling.

## Setup: Import Libraries

In [119]:
import logging
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Tuple, Dict, List
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

print('✓ All libraries imported successfully')

✓ All libraries imported successfully


## Step 1: Load Data

In [120]:
# Define paths
current_dir = Path.cwd()
parquet_path = current_dir.parent / 'output' / 'knee_replacement_providerStep1.parquet'

print(f"Current directory: {current_dir}")
print(f"Looking for parquet file at: {parquet_path}")
print(f"File exists: {parquet_path.exists()}")

if not parquet_path.exists():
    print(f"\n⚠️  File not found at {parquet_path}")
    print(f"Searching for parquet files...")
    for f in current_dir.parent.glob('**/knee_replacement_providerStep1.parquet'):
        parquet_path = f
        print(f"✓ Found: {parquet_path}")
        break

Current directory: c:\Users\GinoH\OneDrive - SGB SMIT Group\Documents\GitHub\EAISI-NHS\notebooks\Lalita\UnderstandingData\Data-prepare
Looking for parquet file at: c:\Users\GinoH\OneDrive - SGB SMIT Group\Documents\GitHub\EAISI-NHS\notebooks\Lalita\UnderstandingData\output\knee_replacement_providerStep1.parquet
File exists: True


In [121]:
# Load the data
df = pd.read_parquet(parquet_path)

print(f"✓ Data loaded successfully")
print(f"  Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nFirst few columns:")
print(df.columns[:20].tolist())

✓ Data loaded successfully
  Shape: 139,236 rows × 82 columns

First few columns:
['Provider Code', 'Procedure', 'Revision Flag', 'Year', 'Age Band', 'Gender', 'Pre-Op Q Assisted', 'Pre-Op Q Assisted By', 'Pre-Op Q Symptom Period', 'Pre-Op Q Previous Surgery', 'Pre-Op Q Living Arrangements', 'Pre-Op Q Disability', 'Heart Disease', 'High Bp', 'Stroke', 'Circulation', 'Lung Disease', 'Diabetes', 'Kidney Disease', 'Nervous System']


## Step 2: Identify Q-Questions and Score Columns

In [122]:
# Identify Pre-Op Q-questions (items 1-12)
pre_op_q_items = [col for col in df.columns if col.startswith('Knee Replacement Pre-Op Q') and 'Score' not in col]
# Remove the composite scores for now
pre_op_q_items = [col for col in pre_op_q_items if col not in ['Knee Replacement Pre-Op Q Score', 'Knee Replacement Pre-Op Q EQ5D Index', 'Knee Replacement Pre-Op Q EQ VAS']]

# Identify Post-Op Q-questions (items 1-12)
post_op_q_items = [col for col in df.columns if col.startswith('Knee Replacement Post-Op Q') and 'Score' not in col]
# Remove the composite scores for now
post_op_q_items = [col for col in post_op_q_items if col not in ['Knee Replacement Post-Op Q Score', 'Knee Replacement Post-Op Q EQ5D Index', 'Knee Replacement Post-Op Q EQ VAS']]

print(f"✓ Pre-Op Q-items (questions 1-12): {len(pre_op_q_items)}")
for col in sorted(pre_op_q_items):
    print(f"  {col}")

print(f"\n✓ Post-Op Q-items (questions 1-12): {len(post_op_q_items)}")
for col in sorted(post_op_q_items):
    print(f"  {col}")

✓ Pre-Op Q-items (questions 1-12): 12
  Knee Replacement Pre-Op Q Confidence
  Knee Replacement Pre-Op Q Kneeling
  Knee Replacement Pre-Op Q Limping
  Knee Replacement Pre-Op Q Night Pain
  Knee Replacement Pre-Op Q Pain
  Knee Replacement Pre-Op Q Shopping
  Knee Replacement Pre-Op Q Stairs
  Knee Replacement Pre-Op Q Standing
  Knee Replacement Pre-Op Q Transport
  Knee Replacement Pre-Op Q Walking
  Knee Replacement Pre-Op Q Washing
  Knee Replacement Pre-Op Q Work

✓ Post-Op Q-items (questions 1-12): 12
  Knee Replacement Post-Op Q Confidence
  Knee Replacement Post-Op Q Kneeling
  Knee Replacement Post-Op Q Limping
  Knee Replacement Post-Op Q Night Pain
  Knee Replacement Post-Op Q Pain
  Knee Replacement Post-Op Q Shopping
  Knee Replacement Post-Op Q Stairs
  Knee Replacement Post-Op Q Standing
  Knee Replacement Post-Op Q Transport
  Knee Replacement Post-Op Q Walking
  Knee Replacement Post-Op Q Washing
  Knee Replacement Post-Op Q Work


## Step 3: Prepare Data for Imputation

We'll impute the individual Q-questions first, then calculate the scores.

In [123]:
# Create a working copy
df_work = df.copy()

# Identify all feature columns (exclude metadata and original scores)
exclude_cols = ['has_missing', 'Knee Replacement Pre-Op Q Score', 'Knee Replacement Post-Op Q Score']
feature_cols = [col for col in df_work.columns if col not in exclude_cols]

# Separate numeric and categorical
numeric_cols = df_work[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_work[feature_cols].select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Feature columns:")
print(f"  Numeric: {len(numeric_cols)}")
print(f"  Categorical: {len(categorical_cols)}")
print(f"  Total: {len(feature_cols)}")

# Extract the Q-question items from feature columns
q_items_to_impute = [col for col in pre_op_q_items + post_op_q_items if col in feature_cols]
print(f"\n✓ Q-items to impute: {len(q_items_to_impute)}")
print(f"  Pre-Op Q-items: {len([col for col in pre_op_q_items if col in feature_cols])} items")
print(f"  Post-Op Q-items: {len([col for col in post_op_q_items if col in feature_cols])} items")

# Other numeric features (not Q-items)
other_numeric = [col for col in numeric_cols if col not in q_items_to_impute]
print(f"✓ Other numeric features: {len(other_numeric)}")

print(f"\n⚠️  Note: Post-Op Q-items will be imputed to calculate the target variable,")
print(f"    but will be EXCLUDED from features (to prevent data leakage)")


Feature columns:
  Numeric: 74
  Categorical: 6
  Total: 80

✓ Q-items to impute: 24
  Pre-Op Q-items: 12 items
  Post-Op Q-items: 12 items
✓ Other numeric features: 50

⚠️  Note: Post-Op Q-items will be imputed to calculate the target variable,
    but will be EXCLUDED from features (to prevent data leakage)


## Step 4: Create Target Variable (Q-Score Difference)

Calculate the difference between post-op and pre-op Q-scores as the target.

In [124]:
print("\n" + "="*70)
print("CALCULATING TARGET VARIABLE: Q-SCORE DIFFERENCE")
print("="*70)

# Get original pre-op and post-op scores (or calculate them)
pre_op_score_orig = df_work['Knee Replacement Pre-Op Q Score'].copy()
post_op_score_orig = df_work['Knee Replacement Post-Op Q Score'].copy()

print(f"\nOriginal Scores:")
print(f"  Pre-Op Q Score: {pre_op_score_orig.notna().sum():,} non-missing values")
print(f"  Post-Op Q Score: {post_op_score_orig.notna().sum():,} non-missing values")

# For now, use the original scores to create the target
# Later we'll recalculate after imputation
y_original = post_op_score_orig - pre_op_score_orig

print(f"\nTarget (Post-Op Q Score - Pre-Op Q Score):")
print(f"  Non-missing values: {y_original.notna().sum():,}")
print(f"  Mean difference: {y_original.mean():.2f}")
print(f"  Std difference: {y_original.std():.2f}")
print(f"  Range: [{y_original.min():.1f}, {y_original.max():.1f}]")

# We'll only use rows where target is not missing
valid_target_idx = y_original.notna()
print(f"\nRows with valid target: {valid_target_idx.sum():,} out of {len(df_work):,}")


CALCULATING TARGET VARIABLE: Q-SCORE DIFFERENCE

Original Scores:
  Pre-Op Q Score: 137,567 non-missing values
  Post-Op Q Score: 136,657 non-missing values

Target (Post-Op Q Score - Pre-Op Q Score):
  Non-missing values: 135,051
  Mean difference: 16.89
  Std difference: 9.86
  Range: [-37.0, 47.0]

Rows with valid target: 135,051 out of 139,236


In [125]:
# Identify columns with >2% missingness (for missingness indicators in pipelines 2 & 3)
print("\n" + "="*70)
print("IDENTIFYING COLUMNS WITH HIGH MISSINGNESS (>2%)")
print("="*70)

# Calculate missingness percentage for all feature columns 
# Exclude post-op Q-items and score columns (these will be removed from final feature set)
cols_to_exclude_from_indicators = post_op_q_items.copy()
cols_to_exclude_from_indicators += ['Pre-Op Q Score Calculated', 'Post-Op Q Score Calculated',
                                     'Knee Replacement Pre-Op Q Score', 'Knee Replacement Post-Op Q Score']

feature_cols_analysis = [col for col in feature_cols if col not in cols_to_exclude_from_indicators]
missing_pcts = {}

for col in feature_cols_analysis:
    missing_pct = 100 * df_work[col].isnull().sum() / len(df_work)
    if missing_pct > 2:
        missing_pcts[col] = missing_pct

# Sort by missingness percentage
cols_with_high_missingness = sorted(missing_pcts.items(), key=lambda x: x[1], reverse=True)

print(f"\n✓ Columns with >2% missingness (in feature set, excluding post-op and score columns):")
if cols_with_high_missingness:
    for col, pct in cols_with_high_missingness:
        print(f"  {col:50s} {pct:6.2f}%")
    print(f"\nTotal: {len(cols_with_high_missingness)} columns")
else:
    print("  None found")

# Store for use in pipelines
cols_for_missingness_indicators = [col for col, _ in cols_with_high_missingness]
print(f"\n✓ These columns will have missingness indicators in Pipelines 2 & 3")


IDENTIFYING COLUMNS WITH HIGH MISSINGNESS (>2%)

✓ Columns with >2% missingness (in feature set, excluding post-op and score columns):
  Knee Replacement EQ VAS_Post-Op Q Predicted         13.95%
  Knee Replacement EQ 5D Index Post-Op Q Predicted    10.29%
  Pre-Op Q EQ5D Index                                  5.39%
  Post-Op Q EQ5D Index                                 4.51%
  Knee Replacement OKS Post-Op Q Predicted             3.86%

Total: 5 columns

✓ These columns will have missingness indicators in Pipelines 2 & 3


## Step 5: Imputation Pipelines

Now we implement the three pipelines with the correct workflow:
1. Use only rows with valid targets
2. Impute Q-question items
3. Calculate the Q-scores from imputed items
4. Create target as the difference
5. Use other features as predictors

### Pipeline 1: Missing Data Preservation

In [126]:
# For Pipeline 1, we keep missing Q-items as-is
# Features = Pre-Op items + other features (with missing values preserved)
# NOTE: Exclude ALL variables with "post" in the name (Post-Op Q-items and any other post-op variables)
# NOTE: Also exclude pre-op score columns to prevent data leakage
# This prevents data leakage since these scores are directly used to calculate the target
cols_to_exclude = post_op_q_items.copy()
# Add any other columns with "post" in the name (case-insensitive)
cols_to_exclude += [col for col in feature_cols if 'post' in col.lower() and col not in cols_to_exclude]
# Add pre-op and post-op score columns (these directly determine the target)
cols_to_exclude += ['Pre-Op Q Score Calculated', 'Post-Op Q Score Calculated', 
                    'Knee Replacement Pre-Op Q Score', 'Knee Replacement Post-Op Q Score']

feature_cols_no_post = [col for col in feature_cols if col not in cols_to_exclude]
X_p1 = df_p1[feature_cols_no_post].copy()

print(f"\n✓ Excluded {len(cols_to_exclude)} columns:")
print(f"  - All post-op variables")
print(f"  - Pre-op and post-op score columns (to prevent target leakage)")
print(f"  Remaining features: {len(feature_cols_no_post)}")

print(f"\n✓ Missing values in Pipeline 1:")
missing_count = X_p1.isnull().sum().sum()
missing_pct = 100 * missing_count / (X_p1.shape[0] * X_p1.shape[1])
print(f"  Total missing: {missing_count:,} ({missing_pct:.2f}%)")

# Train-test split
X_train_p1, X_test_p1, y_train_p1, y_test_p1 = train_test_split(
    X_p1, y_p1, test_size=0.2, random_state=42
)

print(f"\n✓ Train-test split:")
print(f"  Train: {X_train_p1.shape[0]:,} samples")
print(f"  Test: {X_test_p1.shape[0]:,} samples")
print(f"  Features (all post-op and score columns excluded): {X_train_p1.shape[1]}")


✓ Excluded 39 columns:
  - All post-op variables
  - Pre-op and post-op score columns (to prevent target leakage)
  Remaining features: 45

✓ Missing values in Pipeline 1:
  Total missing: 7,127 (0.12%)

✓ Train-test split:
  Train: 108,040 samples
  Test: 27,011 samples
  Features (all post-op and score columns excluded): 45
  Total missing: 7,127 (0.12%)

✓ Train-test split:
  Train: 108,040 samples
  Test: 27,011 samples
  Features (all post-op and score columns excluded): 45


In [127]:
# Save Pipeline 1
output_dir = Path.cwd() / 'prepared_datasets'
output_dir.mkdir(parents=True, exist_ok=True)

X_train_p1.to_parquet(output_dir / "pipeline1_missing_train.parquet")
X_test_p1.to_parquet(output_dir / "pipeline1_missing_test.parquet")
y_train_p1.to_frame(name='Q_Score_Difference').to_parquet(output_dir / "pipeline1_missing_train_target.parquet")
y_test_p1.to_frame(name='Q_Score_Difference').to_parquet(output_dir / "pipeline1_missing_test_target.parquet")

print(f"✓ Pipeline 1 saved to {output_dir}")

✓ Pipeline 1 saved to c:\Users\GinoH\OneDrive - SGB SMIT Group\Documents\GitHub\EAISI-NHS\notebooks\Lalita\UnderstandingData\Data-prepare\prepared_datasets


### Pipeline 2: Median/Mode Imputation (Impute Q-items, then calculate scores)

In [128]:
print("\n" + "="*70)
print("PIPELINE 2: MEDIAN/MODE IMPUTATION")
print("="*70)

# Use same rows as Pipeline 1
df_p2 = df_work.loc[valid_target_idx].copy()
y_p2 = y_original.loc[valid_target_idx].copy()

print(f"\n✓ Starting with {len(df_p2):,} valid rows")

# Create missingness indicators BEFORE train-test split
print(f"\n✓ Creating missingness indicators for {len(cols_for_missingness_indicators)} high-missingness columns...")
for col in cols_for_missingness_indicators:
    df_p2[f'{col}_is_missing'] = df_p2[col].isnull().astype(int)
    print(f"  Created: {col}_is_missing")

# Train-test split FIRST (before imputation to prevent data leakage)
df_train_p2, df_test_p2, y_train_p2, y_test_p2 = train_test_split(
    df_p2, y_p2, test_size=0.2, random_state=42
)

print(f"\n✓ Train-test split (before imputation):")
print(f"  Train: {len(df_train_p2):,} samples")
print(f"  Test: {len(df_test_p2):,} samples")



PIPELINE 2: MEDIAN/MODE IMPUTATION

✓ Starting with 135,051 valid rows

✓ Creating missingness indicators for 5 high-missingness columns...
  Created: Knee Replacement EQ VAS_Post-Op Q Predicted_is_missing
  Created: Knee Replacement EQ 5D Index Post-Op Q Predicted_is_missing
  Created: Pre-Op Q EQ5D Index_is_missing
  Created: Post-Op Q EQ5D Index_is_missing
  Created: Knee Replacement OKS Post-Op Q Predicted_is_missing

✓ Train-test split (before imputation):
  Train: 108,040 samples
  Test: 27,011 samples


### Pipeline 2: Median/Mode Imputation with Missingness Indicators

We'll add binary indicator columns for features with >2% missingness before imputation.


In [129]:
# Step 1: Impute the Q-question items
print(f"\n✓ Imputing Q-question items...")

# Numeric imputer for Q-items
q_items_numeric = [col for col in q_items_to_impute if col in numeric_cols]
numeric_imputer = SimpleImputer(strategy='median')

# Fit on training data only
df_train_p2[q_items_numeric] = numeric_imputer.fit_transform(df_train_p2[q_items_numeric])
df_test_p2[q_items_numeric] = numeric_imputer.transform(df_test_p2[q_items_numeric])

print(f"  Imputed {len(q_items_numeric)} numeric Q-items with median")

# Categorical imputer for any categorical Q-items
q_items_categorical = [col for col in q_items_to_impute if col in categorical_cols]
if q_items_categorical:
    categorical_imputer = SimpleImputer(strategy='most_frequent')
    df_train_p2[q_items_categorical] = categorical_imputer.fit_transform(df_train_p2[q_items_categorical])
    df_test_p2[q_items_categorical] = categorical_imputer.transform(df_test_p2[q_items_categorical])
    print(f"  Imputed {len(q_items_categorical)} categorical Q-items with mode")

# Impute other numeric features
other_numeric_imputer = SimpleImputer(strategy='median')
df_train_p2[other_numeric] = other_numeric_imputer.fit_transform(df_train_p2[other_numeric])
df_test_p2[other_numeric] = other_numeric_imputer.transform(df_test_p2[other_numeric])

print(f"  Imputed {len(other_numeric)} other numeric features with median")

# Impute categorical features
if categorical_cols:
    cat_imputer = SimpleImputer(strategy='most_frequent')
    df_train_p2[categorical_cols] = cat_imputer.fit_transform(df_train_p2[categorical_cols])
    df_test_p2[categorical_cols] = cat_imputer.transform(df_test_p2[categorical_cols])
    print(f"  Imputed {len(categorical_cols)} categorical features with mode")


✓ Imputing Q-question items...
  Imputed 24 numeric Q-items with median
  Imputed 50 other numeric features with median
  Imputed 6 categorical features with mode
  Imputed 50 other numeric features with median
  Imputed 6 categorical features with mode


In [130]:
# Step 2: Recalculate Q-scores from imputed Q-items
print(f"\n✓ Recalculating Q-scores from imputed items...")

# Pre-Op Q Score = sum of pre-op Q-items
pre_op_q_items_in_data = [col for col in pre_op_q_items if col in df_train_p2.columns]
df_train_p2['Pre-Op Q Score Calculated'] = df_train_p2[pre_op_q_items_in_data].sum(axis=1)
df_test_p2['Pre-Op Q Score Calculated'] = df_test_p2[pre_op_q_items_in_data].sum(axis=1)

# Post-Op Q Score = sum of post-op Q-items
post_op_q_items_in_data = [col for col in post_op_q_items if col in df_train_p2.columns]
df_train_p2['Post-Op Q Score Calculated'] = df_train_p2[post_op_q_items_in_data].sum(axis=1)
df_test_p2['Post-Op Q Score Calculated'] = df_test_p2[post_op_q_items_in_data].sum(axis=1)

print(f"  Pre-Op Q Score (calculated): mean = {df_train_p2['Pre-Op Q Score Calculated'].mean():.2f}")
print(f"  Post-Op Q Score (calculated): mean = {df_train_p2['Post-Op Q Score Calculated'].mean():.2f}")

# Recalculate target = Post-Op - Pre-Op
y_train_p2_recalc = df_train_p2['Post-Op Q Score Calculated'] - df_train_p2['Pre-Op Q Score Calculated']
y_test_p2_recalc = df_test_p2['Post-Op Q Score Calculated'] - df_test_p2['Pre-Op Q Score Calculated']

print(f"\n✓ Recalculated target variable:")
print(f"  Train mean difference: {y_train_p2_recalc.mean():.2f}")
print(f"  Test mean difference: {y_test_p2_recalc.mean():.2f}")


✓ Recalculating Q-scores from imputed items...
  Pre-Op Q Score (calculated): mean = 19.01
  Post-Op Q Score (calculated): mean = 35.89

✓ Recalculated target variable:
  Train mean difference: 16.87
  Test mean difference: 16.95


In [131]:
# Step 3: Prepare features (exclude all post-op variables and calculated score columns)
# NOTE: ALL columns with "post" in the name are excluded to prevent data leakage
# NOTE: Pre-op and post-op score columns are excluded (they determine the target)
# NOTE: Missingness indicators are included (created before imputation)
cols_to_exclude = post_op_q_items.copy()
# Add any other columns with "post" in the name (case-insensitive)
cols_to_exclude += [col for col in feature_cols if 'post' in col.lower() and col not in cols_to_exclude]
# Add pre-op and post-op score columns (these directly determine the target)
cols_to_exclude += ['Pre-Op Q Score Calculated', 'Post-Op Q Score Calculated',
                    'Knee Replacement Pre-Op Q Score', 'Knee Replacement Post-Op Q Score']

feature_cols_for_model = [col for col in feature_cols if col not in cols_to_exclude]

# Add missingness indicator columns
missingness_indicator_cols = [f'{col}_is_missing' for col in cols_for_missingness_indicators]
feature_cols_for_model_with_indicators = feature_cols_for_model + missingness_indicator_cols

X_train_p2 = df_train_p2[feature_cols_for_model_with_indicators].copy()
X_test_p2 = df_test_p2[feature_cols_for_model_with_indicators].copy()

print(f"✓ Feature set prepared:")
print(f"  Base features (all post-op and score columns excluded): {len(feature_cols_for_model)}")
print(f"  Missingness indicators: {len(missingness_indicator_cols)}")
print(f"  Total features: {X_train_p2.shape[1]}")
print(f"  Missing values (train): {X_train_p2.isnull().sum().sum()}")
print(f"  Missing values (test): {X_test_p2.isnull().sum().sum()}")

✓ Feature set prepared:
  Base features (all post-op and score columns excluded): 45
  Missingness indicators: 5
  Total features: 50
  Missing values (train): 0
  Missing values (test): 0
  Missing values (train): 0
  Missing values (test): 0


In [132]:
# Save Pipeline 2
X_train_p2.to_parquet(output_dir / "pipeline2_median_mode_train.parquet")
X_test_p2.to_parquet(output_dir / "pipeline2_median_mode_test.parquet")
y_train_p2_recalc.to_frame(name='Q_Score_Difference').to_parquet(output_dir / "pipeline2_median_mode_train_target.parquet")
y_test_p2_recalc.to_frame(name='Q_Score_Difference').to_parquet(output_dir / "pipeline2_median_mode_test_target.parquet")

print(f"✓ Pipeline 2 saved")

✓ Pipeline 2 saved


### Pipeline 3: MICE Imputation with Missingness Indicators

We'll add binary indicator columns for features with >2% missingness before imputation.


In [133]:
print("\n" + "="*70)
print("PIPELINE 3: MICE IMPUTATION")
print("="*70)

# Use same rows as others
df_p3 = df_work.loc[valid_target_idx].copy()
y_p3 = y_original.loc[valid_target_idx].copy()

# Create missingness indicators BEFORE train-test split
print(f"\n✓ Creating missingness indicators for {len(cols_for_missingness_indicators)} high-missingness columns...")
for col in cols_for_missingness_indicators:
    df_p3[f'{col}_is_missing'] = df_p3[col].isnull().astype(int)

# Train-test split
df_train_p3, df_test_p3, y_train_p3, y_test_p3 = train_test_split(
    df_p3, y_p3, test_size=0.2, random_state=42
)

print(f"\n✓ Train-test split:")
print(f"  Train: {len(df_train_p3):,} samples")
print(f"  Test: {len(df_test_p3):,} samples")



PIPELINE 3: MICE IMPUTATION

✓ Creating missingness indicators for 5 high-missingness columns...

✓ Train-test split:
  Train: 108,040 samples
  Test: 27,011 samples

✓ Train-test split:
  Train: 108,040 samples
  Test: 27,011 samples


In [134]:
# Encode categorical features (MICE needs numeric)
print(f"\n✓ Encoding categorical features...")

df_train_p3_enc = df_train_p3.copy()
df_test_p3_enc = df_test_p3.copy()
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    all_values = pd.concat([df_train_p3[col], df_test_p3[col]], ignore_index=True).astype(str)
    le.fit(all_values)
    
    df_train_p3_enc[col] = le.transform(df_train_p3[col].astype(str))
    df_test_p3_enc[col] = le.transform(df_test_p3[col].astype(str))
    label_encoders[col] = le

print(f"  Encoded {len(categorical_cols)} categorical features")


✓ Encoding categorical features...
  Encoded 6 categorical features
  Encoded 6 categorical features


In [135]:
# Fit MICE on all features (to capture relationships)
print(f"\n✓ Fitting IterativeImputer (MICE)...")
print(f"  This may take 1-2 minutes...\n")

mice_imputer = IterativeImputer(
    max_iter=10,
    random_state=42,
    verbose=0
)

# Fit and transform
X_train_p3_array = mice_imputer.fit_transform(df_train_p3_enc[feature_cols])
X_test_p3_array = mice_imputer.transform(df_test_p3_enc[feature_cols])

print(f"✓ MICE imputation complete!")


✓ Fitting IterativeImputer (MICE)...
  This may take 1-2 minutes...

✓ MICE imputation complete!
✓ MICE imputation complete!


In [136]:
# Convert back to DataFrames
df_train_p3_imputed = pd.DataFrame(X_train_p3_array, columns=feature_cols, index=df_train_p3.index)
df_test_p3_imputed = pd.DataFrame(X_test_p3_array, columns=feature_cols, index=df_test_p3.index)

# Decode categorical features
print(f"\n✓ Decoding categorical features...")

for col in categorical_cols:
    le = label_encoders[col]
    
    # Round and clip to valid range
    train_encoded = df_train_p3_imputed[col].round().astype(int).clip(0, len(le.classes_)-1)
    test_encoded = df_test_p3_imputed[col].round().astype(int).clip(0, len(le.classes_)-1)
    
    # Inverse transform
    df_train_p3_imputed[col] = le.inverse_transform(train_encoded)
    df_test_p3_imputed[col] = le.inverse_transform(test_encoded)

print(f"  Decoded successfully")


✓ Decoding categorical features...
  Decoded successfully


In [137]:
# Recalculate Q-scores from imputed items
print(f"\n✓ Recalculating Q-scores from MICE-imputed items...")

pre_op_q_items_in_data = [col for col in pre_op_q_items if col in df_train_p3_imputed.columns]
post_op_q_items_in_data = [col for col in post_op_q_items if col in df_train_p3_imputed.columns]

df_train_p3_imputed['Pre-Op Q Score Calculated'] = df_train_p3_imputed[pre_op_q_items_in_data].sum(axis=1)
df_test_p3_imputed['Pre-Op Q Score Calculated'] = df_test_p3_imputed[pre_op_q_items_in_data].sum(axis=1)

df_train_p3_imputed['Post-Op Q Score Calculated'] = df_train_p3_imputed[post_op_q_items_in_data].sum(axis=1)
df_test_p3_imputed['Post-Op Q Score Calculated'] = df_test_p3_imputed[post_op_q_items_in_data].sum(axis=1)

# Target = difference
y_train_p3_recalc = df_train_p3_imputed['Post-Op Q Score Calculated'] - df_train_p3_imputed['Pre-Op Q Score Calculated']
y_test_p3_recalc = df_test_p3_imputed['Post-Op Q Score Calculated'] - df_test_p3_imputed['Pre-Op Q Score Calculated']

print(f"  Train mean difference: {y_train_p3_recalc.mean():.2f}")
print(f"  Test mean difference: {y_test_p3_recalc.mean():.2f}")


✓ Recalculating Q-scores from MICE-imputed items...
  Train mean difference: 16.87
  Test mean difference: 16.95


In [138]:
# Prepare features (exclude all post-op variables and score columns)
# NOTE: ALL columns with "post" in the name are excluded to prevent data leakage
# NOTE: Pre-op and post-op score columns are excluded (they determine the target)
# NOTE: Missingness indicators are included (created before imputation)
cols_to_exclude = post_op_q_items.copy()
# Add any other columns with "post" in the name (case-insensitive)
cols_to_exclude += [col for col in feature_cols if 'post' in col.lower() and col not in cols_to_exclude]
# Add pre-op and post-op score columns (these directly determine the target)
cols_to_exclude += ['Pre-Op Q Score Calculated', 'Post-Op Q Score Calculated',
                    'Knee Replacement Pre-Op Q Score', 'Knee Replacement Post-Op Q Score']

feature_cols_for_model = [col for col in feature_cols if col not in cols_to_exclude]

# Add missingness indicator columns
missingness_indicator_cols = [f'{col}_is_missing' for col in cols_for_missingness_indicators]
feature_cols_for_model_with_indicators = feature_cols_for_model + missingness_indicator_cols

X_train_p3 = df_train_p3_imputed[feature_cols_for_model].copy()
X_test_p3 = df_test_p3_imputed[feature_cols_for_model].copy()

# Add missingness indicators from the original (non-encoded) dataframes
for col in cols_for_missingness_indicators:
    X_train_p3[f'{col}_is_missing'] = df_train_p3[f'{col}_is_missing']
    X_test_p3[f'{col}_is_missing'] = df_test_p3[f'{col}_is_missing']

print(f"✓ Feature set prepared:")
print(f"  Base features (all post-op and score columns excluded): {len(feature_cols_for_model)}")
print(f"  Missingness indicators: {len(missingness_indicator_cols)}")
print(f"  Total features: {X_train_p3.shape[1]}")
print(f"  Missing values (train): {X_train_p3.isnull().sum().sum()}")
print(f"  Missing values (test): {X_test_p3.isnull().sum().sum()}")

✓ Feature set prepared:
  Base features (all post-op and score columns excluded): 45
  Missingness indicators: 5
  Total features: 50
  Missing values (train): 0
  Missing values (train): 0
  Missing values (test): 0
  Missing values (test): 0


In [139]:
# Save Pipeline 3
X_train_p3.to_parquet(output_dir / "pipeline3_mice_train.parquet")
X_test_p3.to_parquet(output_dir / "pipeline3_mice_test.parquet")
y_train_p3_recalc.to_frame(name='Q_Score_Difference').to_parquet(output_dir / "pipeline3_mice_train_target.parquet")
y_test_p3_recalc.to_frame(name='Q_Score_Difference').to_parquet(output_dir / "pipeline3_mice_test_target.parquet")

print(f"✓ Pipeline 3 saved")

✓ Pipeline 3 saved


## Summary: All Pipelines Complete

In [140]:
print("\n" + "="*70)
print("ALL PIPELINES COMPLETE")
print("="*70)

# Create comparison
comparison = {
    'Pipeline': ['Pipeline 1 (Missing)', 'Pipeline 2 (Median/Mode)', 'Pipeline 3 (MICE)'],
    'Train Samples': [X_train_p1.shape[0], X_train_p2.shape[0], X_train_p3.shape[0]],
    'Test Samples': [X_test_p1.shape[0], X_test_p2.shape[0], X_test_p3.shape[0]],
    'Features': [X_train_p1.shape[1], X_train_p2.shape[1], X_train_p3.shape[1]],
    'Missing Values (Train)': [
        X_train_p1.isnull().sum().sum(),
        X_train_p2.isnull().sum().sum(),
        X_train_p3.isnull().sum().sum()
    ],
    'Target Mean (Train)': [
        y_train_p1.mean(),
        y_train_p2_recalc.mean(),
        y_train_p3_recalc.mean()
    ]
}

comparison_df = pd.DataFrame(comparison)
print("\n" + comparison_df.to_string(index=False))
print("\n" + "="*70)


ALL PIPELINES COMPLETE

                Pipeline  Train Samples  Test Samples  Features  Missing Values (Train)  Target Mean (Train)
    Pipeline 1 (Missing)         108040         27011        45                    5661            16.873769
Pipeline 2 (Median/Mode)         108040         27011        50                       0            16.873769
       Pipeline 3 (MICE)         108040         27011        50                       0            16.873769



## Output Files Generated

In [141]:
# List all generated files
print(f"\nGenerated files in: {output_dir}\n")

output_files = sorted(output_dir.glob('*.parquet'))
for i, f in enumerate(output_files, 1):
    size_kb = f.stat().st_size / 1024
    print(f"{i:2d}. {f.name:50s} ({size_kb:8.1f} KB)")

print(f"\nTotal: {len(output_files)} files")
total_size_mb = sum(f.stat().st_size for f in output_files) / (1024*1024)
print(f"Total size: {total_size_mb:.2f} MB")


Generated files in: c:\Users\GinoH\OneDrive - SGB SMIT Group\Documents\GitHub\EAISI-NHS\notebooks\Lalita\UnderstandingData\Data-prepare\prepared_datasets

 1. pipeline1_missing_test.parquet                     (   561.9 KB)
 2. pipeline1_missing_test_target.parquet              (   197.4 KB)
 3. pipeline1_missing_train.parquet                    (  2174.3 KB)
 4. pipeline1_missing_train_target.parquet             (   798.5 KB)
 5. pipeline2_median_mode_test.parquet                 (   578.8 KB)
 6. pipeline2_median_mode_test_target.parquet          (   197.4 KB)
 7. pipeline2_median_mode_train.parquet                (  2228.5 KB)
 8. pipeline2_median_mode_train_target.parquet         (   798.5 KB)
 9. pipeline3_mice_test.parquet                        (   601.1 KB)
10. pipeline3_mice_test_target.parquet                 (   197.4 KB)
11. pipeline3_mice_train.parquet                       (  2311.2 KB)
12. pipeline3_mice_train_target.parquet                (   798.5 KB)

Total: 12 files

### Pipeline Descriptions

**Pipeline 1: Missing Data Preservation**
- No imputation
- Missing values retained as-is (for tree-based models)
- No missingness indicators

**Pipeline 2: Median/Mode Imputation**
- SimpleImputer with median (numeric) and mode (categorical)
- Imputes all missing values
- **Includes missingness indicator columns** for features with >2% missingness
- Indicators capture the original missing pattern for modeling

**Pipeline 3: MICE Imputation**
- IterativeImputer with Bayesian Ridge (10 iterations)
- Imputes based on relationships between features
- **Includes missingness indicator columns** for features with >2% missingness
- Indicators capture the original missing pattern for modeling

### Imputation Workflow
1. **Identify Q-items** (pre-op and post-op questions)
2. **Create missingness indicators** (Pipelines 2 & 3 only, for columns with >2% missing)
3. **Impute Q-items** (both pre-op AND post-op, needed to calculate target)
4. **Impute other features** (all remaining missing values)
5. **Recalculate Q-scores** by summing the imputed items
6. **Create target** as the difference between post and pre scores
7. **Remove ALL post-op variables and score columns** (data leakage prevention)
8. **Prepare features** for modeling:
   - **INCLUDE:** Pre-Op features (individual items only, NOT pre-op score) + other non-post features + missingness indicators
   - **EXCLUDE:** ALL post-op variables + pre-op/post-op score columns (data leakage prevention)

### Why Exclude ALL Post-Op Variables AND Score Columns?
- **Post-op variables:** Directly determine the target (post-op Q-Score is the numerator of the difference)
- **Pre-op Q-Score:** Used to calculate the target (is the denominator of the difference)
- **Post-op Q-Score:** The primary component of the target variable
- Including ANY of these means the model has direct access to what it's trying to predict
- This is **severe data leakage** - the model would achieve artificially perfect accuracy
- The individual pre-op Q-items are retained since they represent patient baseline characteristics
- At prediction time, we CANNOT have post-op information (it occurs after surgery)
- At prediction time, we CANNOT have pre-op Q-Score (only individual item responses are available)
- Therefore, models must learn from pre-op item responses and patient characteristics only

### Data Leakage Prevention
- Train-test split is done **before** imputation AND before creating indicators
- Imputers are fit **only on training data**
- Test data uses imputation parameters from training
- **ALL post-op variables AND score columns are removed** before any model training
- Missingness indicators are created from original data before imputation
- This ensures realistic model evaluation and future applicability
- At prediction time on new patients, only pre-op item responses will be available

### Examples of Excluded Variables
**Post-Op Variables:**
- Post-Op Q-items (Q1-Q12 post-op questions)
- Post-Op Q Score
- Any other columns with "post" in the name (case-insensitive)

**Score Columns:**
- Pre-Op Q Score (calculated from individual items during preprocessing)
- Post-Op Q Score (calculated from individual items during preprocessing)
- Knee Replacement Pre-Op Q Score (original data)
- Knee Replacement Post-Op Q Score (original data)
